In [ ]:
import pandas as pd
import json

import plotly.express as px

In [ ]:
def get_movie_pool(filtered_df, genre_ids=None, year_range=None, director_ids=None):
    df = filtered_df
    if genre_ids is not None:
        df = df[df['main_genre_id'].isin(genre_ids)]
    if year_range is not None:
        df = df[(df['release_year'] >= year_range[0]) & (df['release_year'] <= year_range[1])]
    if director_ids is not None:
        df = df[df['director_id'].isin(director_ids)]
    return df

def recommend_movies_for_user(
    user_id,
    filtered_df,
    df_ratings,
    df_users,
    n_recs=5,
    explain=True,
    overview_topic_cols=None,
    topic_words=None,
    tag_topic_cols=None,
    tag_topic_words=None,
):
    seen = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    pool = filtered_df[~filtered_df['movieId'].isin(seen)].copy()
    if pool.empty:
        return []

    user_row = df_users[df_users['userId'] == user_id]
    user_top_genres = [user_row['top_genre_1_id'].values[0], user_row['top_genre_2_id'].values[0]]

    pool['score'] = 0
    pool.loc[pool['main_genre_id'].isin(user_top_genres), 'score'] += 1
    if 'movie_cluster' in pool.columns and 'user_cluster' in user_row.columns:
        pool.loc[pool['movie_cluster'] == user_row['user_cluster'].values[0], 'score'] += 1
    pool['score'] += pool['popularity_score'].rank(pct=True)
    pool['score'] += pool['critical_success'].rank(pct=True)
    pool['score'] += pool['vote_average'].rank(pct=True)

    pool = pool.sort_values('score', ascending=False)
    recs = pool.head(n_recs)

    explanations = []
    for _, row in recs.iterrows():
        why = []
        if row['main_genre_id'] in user_top_genres:
            why.append(f"Matches your favorite genre ({row['main_genre']})")
        if 'movie_cluster' in pool.columns and row['movie_cluster'] == user_row['user_cluster'].values[0]:
            why.append(f"In your preferred cluster (based on similar movies)")
        if row['popularity_score'] > pool['popularity_score'].median():
            why.append("Popular among other users")
        if row['critical_success'] > pool['critical_success'].median():
            why.append("Critically acclaimed")
        # Top topics/keywords/themes
        topic_strs = []
        if overview_topic_cols is not None and topic_words is not None:
            theme_words = get_top_topic_words(row, overview_topic_cols, topic_words, n=2)
            if theme_words:
                topic_strs.append("Overview: " + theme_words)
        else:
            print(overview_topic_cols)
        if tag_topic_cols is not None and tag_topic_words is not None:
            tag_words = get_top_topic_words(row, tag_topic_cols, tag_topic_words, n=2)
            if tag_words:
                topic_strs.append("Tags: " + tag_words)
        else:
            print(tag_topic_cols)
        if topic_strs:
            why.append("Notable themes: " + " | ".join(topic_strs))
        explanations.append({
            "movieId": row['movieId'],
            "title": row['title'],
            "explanation": "; ".join(why)
        })

    if explain:
        return explanations
    else:
        return recs[['movieId', 'title']]

def get_top_topic_words(row, topic_cols, topic_words, n=2):
    """Return the most salient topics/words for this row."""
    # Get the topic indices sorted by strength
    topic_strengths = [(i, row[topic_col]) for i, topic_col in enumerate(topic_cols)]
    topic_strengths.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in topic_strengths[:n]]
    words = []
    for idx in top_indices:
        words.extend(topic_words.get(idx, []))
    return ', '.join(words[:8])

In [ ]:
df_movies = pd.read_parquet("/Users/aniket/TU_Eindhoven/2_Study/Q4_2AMV10_Visual_Analytics/4_Code/2_AMV10_Visual_Analytics/data/1_movies_data_for_app.parquet")
df_users = pd.read_parquet("/Users/aniket/TU_Eindhoven/2_Study/Q4_2AMV10_Visual_Analytics/4_Code/2_AMV10_Visual_Analytics/data/1_all_users_stats_with_clusters.parquet")
df_ratings = pd.read_parquet("/Users/aniket/TU_Eindhoven/2_Study/Q4_2AMV10_Visual_Analytics/4_Code/2_AMV10_Visual_Analytics/data/1_ratings_data_filtered.parquet")

In [ ]:
all_genres_from_list = set(g for genres in df_movies['genre_list'] for g in genres)
all_genres_from_main = set(df_movies['main_genre'].dropna().unique())
all_genres = all_genres_from_list | all_genres_from_main
genre2id = {genre: idx for idx, genre in enumerate(sorted(all_genres))}
id2genre = {idx: genre for genre, idx in genre2id.items()}

all_directors = df_movies['director'].dropna().unique()
director2id = {director: idx for idx, director in enumerate(sorted(all_directors))}
id2director = {idx: director for director, idx in director2id.items()}

all_actors = df_movies['lead_actor'].dropna().unique()
actor2id = {actor: idx for idx, actor in enumerate(sorted(all_actors))}
id2actor = {idx: actor for actor, idx in actor2id.items()}

In [ ]:
core_num_columns = [
    'popularity',
    'vote_average',
    'vote_count',
    'critical_success',
    'runtime_bin_id',
    'release_decade',
    # 'release_year',
    "main_genre_id"

] # + [col for col in df_movies.columns if col.startswith('main_genre_')]
overview_topic_cols = [col for col in df_movies.columns if col.startswith("overview_topic_")]
tag_topic_cols = [col for col in df_movies.columns if col.startswith("tag_topic_")]
keywords_topic_cols = [col for col in df_movies.columns if col.startswith("keyword_topic_")]
core_num_columns += [col for col in df_movies.columns if col.startswith('runtime_bin_')]

core_num_columns += overview_topic_cols
# core_num_columns += tag_topic_cols
core_num_columns += keywords_topic_cols

core_num_columns += [
    'lead_actor_popularity',
    'director_popularity'
]

core_num_columns += [
    'director_cluster', 
    'actor_cluster',
]

In [ ]:
with open('../data/topic_words.json', 'r') as f:
    topic_dicts = json.load(f)
    topic_words = topic_dicts['overview_topic_words']
    topic_words = {int(k): v for k, v in topic_words.items()}
    tag_topic_words = topic_dicts['tag_topic_words']
    tag_topic_words = {int(k): v for k, v in tag_topic_words.items()}
    keywords_topic_words = topic_dicts['keywords_topic_words']
    keywords_topic_words = {int(k): v for k, v in keywords_topic_words.items()}

In [ ]:
filtered_df = df_movies.copy()
current_user = df_users.copy()
filtered_ratings = df_ratings.copy()

In [ ]:
selected_genres = [genre2id['Animation'], genre2id['Comedy']]
filtered_pool = get_movie_pool(filtered_df, genre_ids=selected_genres)

recs = recommend_movies_for_user(
    user_id=4,
    filtered_df=filtered_pool,
    df_ratings=df_ratings,
    df_users=df_users,
    n_recs=5,
    explain=True,
    overview_topic_cols=overview_topic_cols,
    topic_words=topic_words,
    tag_topic_cols=tag_topic_cols,
    tag_topic_words=tag_topic_words
)
for rec in recs:
    print(f"Recommended: {rec['title']}\nWhy: {rec['explanation']}\n")

In [ ]:
scatter_df = filtered_df.copy()

fig = px.scatter(
    scatter_df, x='pca_1', y='pca_2',
    color='movie_cluster',
    hover_data=['title', 'main_genre', 'director', 'lead_actor', 'vote_average', 'popularity_score'],
    title="Movie Clusters in PCA Space"
)
fig.show()

In [ ]:
user_id = 4

watched = df_ratings[df_ratings['userId'] == user_id][['movieId', 'rating']]
watched = watched.loc[watched["movieId"].isin(filtered_df["movieId"])]
watched = watched.merge(filtered_pool[['movieId', 'title', 'main_genre', 'pca_1', 'pca_2', 'movie_cluster']], on='movieId', how='inner')
watched['status'] = 'Watched'

recommended_ids = [rec['movieId'] for rec in recs]
recommended = filtered_pool[filtered_pool['movieId'].isin(recommended_ids)].copy()
recommended['status'] = 'Recommended'
recommended['rating'] = recommended["vote_average"].values

plot_df = pd.concat([watched, recommended], ignore_index=True)


plot_df = pd.concat([watched, recommended], ignore_index=True)
# Optionally, add a new column for easier color/symbol mapping
plot_df['status'] = plot_df['status'].astype(str)

PLOT_COLUMNS = [
    'movieId', 'title', 'main_genre', 'rating', 'status', 
    'pca_1', 'pca_2', 'movie_cluster'
]
plot_df = plot_df[PLOT_COLUMNS]

In [ ]:
plot_columns = [
    'movieId', 'title', 'main_genre', 'rating', 'status', 
    'pca_1', 'pca_2', 'movie_cluster'
]
plot_df_for_vis = plot_df[plot_columns]

fig = px.scatter(
    plot_df_for_vis,
    x='pca_1', y='pca_2',
    color='status',
    symbol='status',
    size='rating',
    hover_data=['title', 'main_genre', 'rating', 'movie_cluster'],
    # facet_col='status',
    title='Watched & Recommended Movies for User 4 in Feature Space'
)
fig.show()


In [ ]:
bar_df = plot_df_for_vis.copy()
bar_df['User Rated'] = bar_df['status'] == 'Watched'

fig = px.bar(
    bar_df,
    x='main_genre', y='rating', color='status',
    barmode='group',
    title='Watched vs Recommended: Average Rating per Genre'
)
fig.show()